In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
methods_names = {
    "dist_linguistic_confidence": "Dist. Ling. Conf.",
    "dist_semantic_uncertainty": "Dist. Semantic Unc.",
    "dist_lnll": "Dist. Token Prob"
}

dataset_size = {
    "mmlu": 14000 ,
    "squadv2": 11900,
    "truthful_qa": 817
}

dataset_map = {
    "mmlu": "MMLU",
    "squadv2": "SQuAD2.0",
    "truthful_qa": "TruthfulQA"
}

model_name_map = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B-Inst.",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B-Inst.",
    "Qwen3-8B": "Qwen3-8B-Inst.",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B-Inst.",
    "gpt-oss-20b": "GPT-OSS-20B",
    "gpt-oss-120b": "GPT-OSS-120B",
    "gemma-4-31B-it": "Gemma-4-31B-It.",
    "Qwen3-235B-A22B-Instruct-2507-tput": "Qwen3-235B-Inst."
}

In [3]:
pct = True

# Cross domain data

In [4]:
prompt_type = "direct_qa"

In [5]:
results_dir = f"/hdd/ivny/{prompt_type}_cross_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    if model_name == "gpt-oss-120b" or model_name.lower() == "qwen3-235b-a22b-instruct-2507-tput":
        continue
    training_set, test_set = dataset_name.split("--")
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["model"] = model_name_map.get(model_name, model_name)
            record["training_set"] = dataset_map.get(training_set, training_set)
            record["test_set"] = dataset_map.get(test_set, test_set)
            record["dataset_size"] = dataset_size.get(test_set, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

Qwen3-8B truthful_qa--mmlu missing calibration_performance.csv


In [6]:
df = pd.DataFrame(all_records).drop(columns=["model", "dataset_size"])
dataset_weighted_average_mean = df.groupby(["training_set", "test_set"]).mean().reset_index()

In [7]:
dataset_weighted_average_mean

,training_set,test_set,original_lc_generalised_ECE,original_lc_faithfulness_divergence,original_lc_ece_mean,original_lc_dAUROC,original_lc_auroc_mean,original_tp_generalised_ECE,original_tp_faithfulness_divergence,original_tp_ece_mean,...,calibrated_tp_rewritten_lc_generalised_ECE,calibrated_tp_rewritten_lc_faithfulness_divergence,calibrated_tp_rewritten_lc_ece_mean,calibrated_tp_rewritten_lc_dAUROC,calibrated_tp_rewritten_lc_auroc_mean,calibrated_su_rewritten_lc_generalised_ECE,calibrated_su_rewritten_lc_faithfulness_divergence,calibrated_su_rewritten_lc_ece_mean,calibrated_su_rewritten_lc_dAUROC,calibrated_su_rewritten_lc_auroc_mean
0,MMLU,SQuAD2.0,0.316237,1.845457,0.291094,0.502220,0.497768,0.282188,50.456940,0.278050,...,0.194536,0.874089,0.160688,0.56154,0.578106,0.218843,1.178547,0.212961,0.586240,0.601872
1,MMLU,TruthfulQA,0.367805,1.837393,0.359136,0.572580,0.608721,0.233883,16.034569,0.229521,...,0.229751,0.801859,0.197948,0.56774,0.606439,0.243806,1.048011,0.238007,0.635800,0.663407
2,SQuAD2.0,MMLU,0.179362,0.843324,0.127745,0.517540,0.533036,0.237855,135.704602,0.225161,...,0.137265,0.566553,0.106952,0.52650,0.537129,0.125378,0.473303,0.089345,0.597040,0.629327
3,SQuAD2.0,TruthfulQA,0.367638,1.837393,0.359136,0.578220,0.608721,0.233793,16.034569,0.229521,...,0.141951,0.544282,0.108188,0.58532,0.623473,0.184421,0.633175,0.170305,0.630860,0.669965
4,TruthfulQA,MMLU,0.349055,1.089448,0.257598,0.472825,0.447745,0.227048,113.748684,0.212130,...,0.134107,0.565597,0.110716,0.51835,0.532126,0.128069,0.426074,0.103909,0.584175,0.623208
5,TruthfulQA,SQuAD2.0,0.316192,1.845457,0.291094,0.501440,0.497768,0.282201,50.456940,0.278050,...,0.162427,0.660642,0.146292,0.53306,0.549772,0.136410,0.620896,0.121863,0.558020,0.576693


In [8]:
dataset_weighted_average_mean.columns.tolist()

['training_set',
 'test_set',
 'original_lc_generalised_ECE',
 'original_lc_faithfulness_divergence',
 'original_lc_ece_mean',
 'original_lc_dAUROC',
 'original_lc_auroc_mean',
 'original_tp_generalised_ECE',
 'original_tp_faithfulness_divergence',
 'original_tp_ece_mean',
 'original_tp_dAUROC',
 'original_tp_auroc_mean',
 'original_su_generalised_ECE',
 'original_su_faithfulness_divergence',
 'original_su_ece_mean',
 'original_su_dAUROC',
 'original_su_auroc_mean',
 'calibrated_lc_generalised_ECE',
 'calibrated_lc_faithfulness_divergence',
 'calibrated_lc_ece_mean',
 'calibrated_lc_dAUROC',
 'calibrated_lc_auroc_mean',
 'calibrated_tp_generalised_ECE',
 'calibrated_tp_faithfulness_divergence',
 'calibrated_tp_ece_mean',
 'calibrated_tp_dAUROC',
 'calibrated_tp_auroc_mean',
 'calibrated_su_generalised_ECE',
 'calibrated_su_faithfulness_divergence',
 'calibrated_su_ece_mean',
 'calibrated_su_dAUROC',
 'calibrated_su_auroc_mean',
 'calibrated_lc_rewritten_lc_generalised_ECE',
 'calibrate

In [9]:
fd_improvement_df = dataset_weighted_average_mean[["training_set", "test_set"]].copy()


fd_improvement_df["Linguistic Confidence"] = (dataset_weighted_average_mean["calibrated_lc_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 
fd_improvement_df["Token Probability"] = (dataset_weighted_average_mean["calibrated_tp_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 
fd_improvement_df["Semantic Uncertainty"] = (dataset_weighted_average_mean["calibrated_su_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 

if pct:
    fd_improvement_df["Linguistic Confidence"] = fd_improvement_df["Linguistic Confidence"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]
    fd_improvement_df["Token Probability"] = fd_improvement_df["Token Probability"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]
    fd_improvement_df["Semantic Uncertainty"] = fd_improvement_df["Semantic Uncertainty"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]

fd_improvement_df.to_dict("records")

[{'training_set': 'MMLU',
  'test_set': 'SQuAD2.0',
  'Linguistic Confidence': -0.34590857822049936,
  'Token Probability': -0.5263564184120605,
  'Semantic Uncertainty': -0.3613789695632327},
 {'training_set': 'MMLU',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.29584199545565554,
  'Token Probability': -0.5635889555081973,
  'Semantic Uncertainty': -0.42962035019685696},
 {'training_set': 'SQuAD2.0',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.42663770461121203,
  'Token Probability': -0.32819044191106767,
  'Semantic Uncertainty': -0.4387648543845944},
 {'training_set': 'SQuAD2.0',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.5984709826018371,
  'Token Probability': -0.7037750916327359,
  'Semantic Uncertainty': -0.6553950710743374},
 {'training_set': 'TruthfulQA',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.2700819004195849,
  'Token Probability': -0.4808405096968605,
  'Semantic Uncertainty': -0.6089078944171579},
 {'training_set': 'Truthf

In [10]:
ece_improvement_df  = dataset_weighted_average_mean[["training_set", "test_set"]].copy()

ece_improvement_df["Linguistic Confidence"] = (dataset_weighted_average_mean["calibrated_lc_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 
ece_improvement_df["Token Probability"] = (dataset_weighted_average_mean["calibrated_tp_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 
ece_improvement_df["Semantic Uncertainty"] = (dataset_weighted_average_mean["calibrated_su_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 

if pct:
    ece_improvement_df["Linguistic Confidence"] = ece_improvement_df["Linguistic Confidence"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]
    ece_improvement_df["Token Probability"] = ece_improvement_df["Token Probability"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]
    ece_improvement_df["Semantic Uncertainty"] = ece_improvement_df["Semantic Uncertainty"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]

ece_improvement_df.to_dict("records")

[{'training_set': 'MMLU',
  'test_set': 'SQuAD2.0',
  'Linguistic Confidence': -0.2071180368776043,
  'Token Probability': -0.384842131774417,
  'Semantic Uncertainty': -0.30797785801608213},
 {'training_set': 'MMLU',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.12208086285035549,
  'Token Probability': -0.3753451910224503,
  'Semantic Uncertainty': -0.33713132645519306},
 {'training_set': 'SQuAD2.0',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.2194102145050023,
  'Token Probability': -0.23470460135230803,
  'Semantic Uncertainty': -0.3009746383824319},
 {'training_set': 'SQuAD2.0',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.3498897536135446,
  'Token Probability': -0.6138834280419653,
  'Semantic Uncertainty': -0.4983611813871417},
 {'training_set': 'TruthfulQA',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.26247173514498684,
  'Token Probability': -0.6157990367263855,
  'Semantic Uncertainty': -0.6330984850966717},
 {'training_set': 'Truthfu

# In domain diagonal data

In [11]:
results_dir = f"/hdd/ivny/{prompt_type}_in_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    if model_name == "gpt-oss-120b" or model_name.lower() == "qwen3-235b-a22b-instruct-2507-tput":
        continue
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["dataset"] = dataset_map.get(dataset_name, dataset_name)
            record["model"] = model_name_map.get(model_name, model_name)
            record["dataset_size"] = dataset_size.get(dataset_name, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

In [12]:
full_df = pd.DataFrame(all_records)
full_df = full_df.drop(columns=["model", "dataset_size"]).groupby("dataset").mean().reset_index()

in_domain_ece = []
in_domain_fd = []

for _, row in full_df.iterrows():
    if pct:
        ece = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': (row["calibrated_lc_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE'],
            'Token Probability': (row["calibrated_tp_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE'],
            'Semantic Uncertainty': (row["calibrated_su_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE']
        }

        fd = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': (row["calibrated_lc_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence'],
            'Token Probability': (row["calibrated_tp_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence'],
            'Semantic Uncertainty': (row["calibrated_su_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence']
        }
    else:
        ece = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': row["calibrated_lc_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE'],
            'Token Probability': row["calibrated_tp_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE'],
            'Semantic Uncertainty': row["calibrated_su_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']
        }

        fd = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': row["calibrated_lc_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence'],
            'Token Probability': row["calibrated_tp_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence'],
            'Semantic Uncertainty': row["calibrated_su_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']
        }
    in_domain_ece.append(ece)
    in_domain_fd.append(fd)


In [13]:
full_df

,dataset,original_lc_generalised_ECE,original_lc_faithfulness_divergence,original_lc_ece_mean,original_lc_dAUROC,original_lc_auroc_mean,original_tp_generalised_ECE,original_tp_faithfulness_divergence,original_tp_ece_mean,original_tp_dAUROC,...,calibrated_su_rewritten_lc_generalised_ECE,calibrated_su_rewritten_lc_faithfulness_divergence,calibrated_su_rewritten_lc_ece_mean,calibrated_su_rewritten_lc_dAUROC,calibrated_su_rewritten_lc_auroc_mean,calibrated_su_beta_guided_lc_generalised_ECE,calibrated_su_beta_guided_lc_faithfulness_divergence,calibrated_su_beta_guided_lc_ece_mean,calibrated_su_beta_guided_lc_dAUROC,calibrated_su_beta_guided_lc_auroc_mean
0,MMLU,0.206484,0.930421,0.155329,0.51433,0.512768,0.237828,135.725216,0.225188,0.58706,...,0.103655,0.729286,0.072451,0.60802,0.635533,0.115144,0.601318,0.089088,0.55980,0.579872
1,SQuAD2.0,0.316164,1.845539,0.291045,0.50810,0.497672,0.282153,50.412510,0.277984,0.61829,...,0.130709,0.627525,0.122142,0.57194,0.588115,0.196288,0.845094,0.187362,0.51722,0.523716
2,TruthfulQA,0.368916,1.840316,0.360211,0.57740,0.610017,0.234719,16.062272,0.230363,0.59648,...,0.202738,0.624315,0.197691,0.61880,0.663677,0.258408,0.811971,0.254689,0.59272,0.630979


In [27]:
in_domain_ece_df = pd.DataFrame(in_domain_ece)
in_domain_fd_df = pd.DataFrame(in_domain_fd)

# Format latex table

In [28]:
def generate_latex_table(faithfulness_data, ece_data, pct):
    def fmt(value, pct=False):
        if value is None:
            return r"-"
        if pct:
            pct = value * 100
            color = "green!70!black" if pct < 0 else "red!70!black"
            abs_pct = abs(pct)
            return rf"\textcolor{{{color}}}{{$\Delta${abs_pct:.2f}\%}}"
        else:
            color = "green!70!black" if value < 0 else "red!70!black"
            abs_value = abs(value)
            return rf"\textcolor{{{color}}}{{$\Delta${abs_value:.4f}}}"

    def lookup(data, train, test):
        for row in data:
            if row["training_set"] == train and row["test_set"] == test:
                return row
        return None

    estimators = [
        ("Linguistic\\\\Confidence", "Linguistic Confidence"),
        ("Token\\\\Probability",     "Token Probability"),
        ("Semantic\\\\Uncertainty",  "Semantic Uncertainty"),
    ]
    datasets = ["MMLU", "SQuAD2.0", "TruthfulQA"]

    def build_block(data, metric_label):
        lines = []
        lines.append(rf"\multirow{{9}}{{=}}{{\textbf{{{metric_label}}}}}")

        for ei, (est_display, est_key) in enumerate(estimators):
            lines.append(rf"& \multirow{{3}}{{=}}{{{est_display}}}")

            for ti, train in enumerate(datasets):
                cells = []
                for col in datasets:
                    # if col == train:
                    #     cells.append("-")
                    # else:
                    row = lookup(data, train, col)
                    cells.append(fmt(row.get(est_key), pct=pct) if row else "-")

                cell_str = " & ".join(cells)

                if ti == 0:
                    lines.append(rf"& {train} & {cell_str} \\")
                else:
                    lines.append(rf"& & {train} & {cell_str} \\")

            if ei < len(estimators) - 1:
                lines.append(r"\cmidrule(lr){2-6}")

        return lines

    if pct:
        latex_lines = [
            r"\begin{table}[t]",
            r"\centering",
            r"\caption{In-domain and cross-domain linguistic-space calibration metric percentage changes for both Faithfulness Divergence and generalised ECE. We report percentage change relative to the pre-calibration metrics. Green text indicates calibration improvement (lower error), whereas red text indicates calibration deterioration (higher error). Averaged across all models, token probability and semantic uncertainty exhibits strong transferability for improving faithfulness and calibration, whilst linguistic confidence shows more limited effectiveness.}",
            r"\label{tab:cross-domain-calibration}",
            r"\small",
            r"\begin{tabular}{p{2cm}p{2cm}lccc}",
            r"\toprule",
            r"\textbf{Metric} & \textbf{Estimator} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\",
            r"\midrule",
        ]
    else:
        latex_lines = [
            r"\begin{table}[t]",
            r"\centering",
            r"\caption{In-domain and cross-domain linguistic-space calibration metric changes for both Faithfulness Divergence and generalised ECE. We report the value change relative to the pre-calibration metrics. Green text indicates calibration improvement (lower error), whereas red text indicates calibration deterioration (higher error). Averaged across all models, token probability and semantic uncertainty exhibits strong transferability for improving faithfulness and calibration, whilst linguistic confidence shows more limited effectiveness.}",
            r"\label{tab:cross-domain-calibration}",
            r"\small",
            r"\begin{tabular}{p{2cm}p{2cm}lccc}",
            r"\toprule",
            r"\textbf{Metric} & \textbf{Estimator} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\",
            r"\midrule",
        ]

    latex_lines.extend(build_block(faithfulness_data, "Faithfulness\\\\Divergence\\\\Mean\\\\Reduction"))
    latex_lines.append(r"\midrule")
    latex_lines.extend(build_block(ece_data, "Generalised\\\\ECE\\\\Mean\\\\Reduction"))

    latex_lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]

    return "\n".join(latex_lines)

print(generate_latex_table(fd_improvement_df.to_dict("records") + in_domain_fd_df.to_dict("records"), 
                           ece_improvement_df.to_dict("records") + in_domain_ece_df.to_dict("records"), 
                           pct=pct))

\begin{table}[t]
\centering
\caption{In-domain and cross-domain linguistic-space calibration metric percentage changes for both Faithfulness Divergence and generalised ECE. We report percentage change relative to the pre-calibration metrics. Green text indicates calibration improvement (lower error), whereas red text indicates calibration deterioration (higher error). Averaged across all models, token probability and semantic uncertainty exhibits strong transferability for improving faithfulness and calibration, whilst linguistic confidence shows more limited effectiveness.}
\label{tab:cross-domain-calibration}
\small
\begin{tabular}{p{2cm}p{2cm}lccc}
\toprule
\textbf{Metric} & \textbf{Estimator} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\
\midrule
\multirow{9}{=}{\textbf{Faithfulness\\Divergence\\Mean\\Reduction}}
& \multirow{3}{=}{Linguistic\\Confidence}
& MMLU & \textcolor{green!70!black}{$\Delta$11.36\%} & \textcolor{green!70!black}{$\Delta$34.59\%} & \textcolor{gre